## Kontrol Yapıları ve Fonksiyonlar

- Motivasyon: Buradaki 'bazı' konular (örn. karar yapıları/döngüler/fonksiyon tanımlama) aslında başka programlama dillerini çalışırken defalarca gördüğüm ve aşina olduğum şeyler. Yine de bildiklerimi cilalamak ve Python syntaxına daha da aşinalık kazanmak için pratiklerini yapmamım iyi olacağını düşündüm.
- Soruları nereden bulduğumu, diğer detayları 'p2_veri_tipleri' dosyasında anlatmıştım. Soruların zorluklarını 5 üzerinden puanlıyorum aynı şekilde
- Closures, Decorators gibi daha zor konuları diğer 'p3' dosyasında çalıştım. 

### 1 - Koşullu İfadeler (1/5)

Bağlam: Bir web uygulamasının yetkilendirme (authorization) modülünü yazıyoruz. Sisteme giriş yapmaya çalışan kullanıcıların hesap durumlarına ve atanan rollerine göre farklı sayfalara yönlendirilmesi veya engellenmesi gerekiyor. Gereksinimler:
1. erisim_kontrolu(rol: str, hesap_aktif_mi: bool) adında bir fonksiyon tanımla.
2. Önceki konularda öğrendiğimiz "truthiness" kuralını kullanarak, eğer rol boş bir string ( "" ) ise if not ile yakalayıp "Rol belirtilmedi" hatası döndür (erken çıkış / early return).
3. Hesabın aktif olup olmadığını kontrol et; hesap pasifse ( False ), rolün ne olduğuna bakılmaksızın doğrudan "Hesap askıda" uyarısı döndür.
4. Eğer hesap aktifse, if/elif/else zinciri kurarak sırasıyla "admin" için tam yetki, "editor" için içerik yetkisi, "standart" için okuma yetkisi mesajları döndür. Gelen rol metnini önceki konulardaki gibi .strip().lower() ile temizle.

In [14]:
# Test Verisi
testler = [
    (" ADMIN ", True),      # Başarılı - Admin (Boşluklu ve büyük harf)     
    ("editor", True),       # Başarılı - Editör
    ("standart", True),     # Başarılı - Standart
    ("misafir", True),      # Başarısız - Bilinmeyen rol
    ("admin", False),       # Başarısız - Rol yetkili ama hesap pasif     
    ("", True)              # Başarısız - Rol boş (Edge-case)
]


def erisim_kontrolu(rol: str, hesap_aktifligi: bool) -> str:
    
    rol_clean = rol.strip().lower()

    # Edge-case 1
    if not rol:
        return "Başarısız - Rol belirtilmedi"
        
    # Edge-case 2
    if not hesap_aktifligi:
        return "Başarısız - Hesap askıda"
        

    if rol_clean == "admin":        return "Başarılı - Tam yetki"
    elif rol_clean == "editor":     return "Başarılı - İçerik yetkisi"
    elif rol_clean == "standart":   return "Başarılı - Okuma yetkisi"
    else:                           return"Başarısız - Bilinmeyen rol"


for rol, hesap_aktifligi in testler:
    sonuc = erisim_kontrolu(rol=rol, hesap_aktifligi=hesap_aktifligi)
    print(f"Rol: {repr(rol)}, Aktif: {hesap_aktifligi} -> {sonuc}")


Rol: ' ADMIN ', Aktif: True -> Başarılı - Tam yetki
Rol: 'editor', Aktif: True -> Başarılı - İçerik yetkisi
Rol: 'standart', Aktif: True -> Başarılı - Okuma yetkisi
Rol: 'misafir', Aktif: True -> Başarısız - Bilinmeyen rol
Rol: 'admin', Aktif: False -> Başarısız - Hesap askıda
Rol: '', Aktif: True -> Başarısız - Rol belirtilmedi


### 2 - Döngüler (3/5)

Bağlam: Bir IoT (Nesnelerin İnterneti) sisteminde, sensörlerden gelen sıcaklık verilerini işleyen ve cihaz bağlantı kopukluklarını yöneten bir kontrol modülü yazıyoruz.
Gereksinimler:
1. İçinde sayısal değerler ve iletişim hatalarını simüle eden None değerleri barındıran bir listeyi for döngüsü ile tara.
2. Döngü içinde, değer None ise veya 0'dan küçük mantıksız bir değerse (sensör hatası) işlemi continue ile atla.
3. Sıcaklık 100.0 derecenin üzerindeyse, yangın riski taşıdığı için döngüyü break ile tamamen durdur ve acil durum bayrağı (flag) döndür.
4. Ayrı bir fonksiyonda, sensör bağlantısı koptuğunda yeniden bağlanmayı denemek için bir while döngüsü kur; maksimum deneme sayısına (örn: 3) ulaşıldığında döngüyü kırıp başarısızlık mesajı ver.
5. Edge-case: Veri listesi boş gönderilirse erken çıkış yaparak uyarı döndür.

In [ ]:
# Test Verisi
test_okumalari = [22.5, None, -5.0, 45.0, 105.5, 30.2]
test_listeleri = [[],[],test_okumalari,[]]


# Gereksinim 1-3 ve 5: --- Sensör Analiz Yapan Fonksiyon ---
def sensor_analizi_yap(veriler: list[float | None]) -> bool:

    if not veriler:         # Edge-case: Boş veri seti kontrolü
        print("Uyarı: İşlenecek sensör verisi bulunamadı.")
        return False

    acil_durum = False
    print("--- Sıcaklık Analizi Başlıyor ---")
    for deger in veriler:

        # Hatalı ölçümleri atla
        if deger is None or deger < 0:
            print(f"Bozuk/eksik veri atlandı: {deger}")
            continue

        # Kritik eşik kontrolü
        if deger > 100.0:
            print(f"Kritik Uyarı! Sıcaklık çok yüksek ({deger}C). Acil duruma geçiliyor.")
            acil_durum = True
            break

        print(f"Normal ölçüm: {deger}C")

    return acil_durum


# Gereksinim 4: --- Yeniden bağlantı deneme Fonksiyonu --- 
def baglanti_dene(veri_seti: list[list[float|None]], max_deneme: int = 3) -> list[float|None] | None:

    deneme = 1
    while deneme <= max_deneme and deneme<=len(veri_seti):
        if not veri_seti[deneme-1]:  
            print(f"Veri bulunamadı! Bağlantı yeniden kuruluyor... \n(Deneme {deneme}/{max_deneme})")
            deneme += 1
        else:
            print(f"Veri bulundu! \n(Deneme {deneme}/{max_deneme})")
            return veri_seti[deneme-1]
        
    if deneme > len(veri_seti):
        print("Kontrol edilecek veri paketi kalmadı.")
    else:
        print("Maksimum deneme sayısına ulaşıldı.")   
    return None


# --- Yazılan Fonksiyonların Sınanması ---
saglam_veriler = baglanti_dene(test_listeleri, max_deneme=3)

if saglam_veriler:
    if sensor_analizi_yap(saglam_veriler):
        print("Acil durum modu aktifleştirilmiş. Sıra dışı değer var.")
    else:
        print("Tüm değerler normal")
else:
    print("Bütün bağlantı denemeleri başarısız olmuş.")


Veri bulunamadı! Bağlantı yeniden kuruluyor... 
(Deneme 1/3)
Veri bulunamadı! Bağlantı yeniden kuruluyor... 
(Deneme 2/3)
Veri bulundu! 
(Deneme 3/3)
--- Sıcaklık Analizi Başlıyor ---
Normal ölçüm: 22.5C
Bozuk/eksik veri atlandı: None
Bozuk/eksik veri atlandı: -5.0
Normal ölçüm: 45.0C
Kritik Uyarı! Sıcaklık çok yüksek (105.5C). Acil duruma geçiliyor.
Acil durum modu aktifleştirilmiş. Sıra dışı değer var.


### 3 - Iterators ve Generators (3/5)

Bağlam: Bir sunucu çiftliğinde günde gigabaytlarca üretilen log (günlük) dosyalarını analiz etmemiz gerekiyor. Dosyanın tamamını bir listeye alıp belleğe (RAM) yüklemek sistemi çökerteceği için, veriyi yalnızca ihtiyaç anında satır satır üreten bellek dostu bir yapı (generator pipeline) kuracağız.
Gereksinimler:
1. Girdi olarak aldığı veriyi (gerçekte dev bir dosya olduğunu varsayalım) satır satır okuyan ve return yerine yield anahtar kelimesini kullanarak bir üreteç (generator) oluşturan log_okuyucu fonksiyonu yaz.
2. İlk üreteçten gelen akışı (iterator) girdi olarak alan ve yalnızca içinde "ERROR" kelimesi geçen satırları yield eden ikinci bir hata_filtresi üreteci yazarak bir "veri boru hattı (pipeline)" kur.
3. Ana programda, arka planda bu sistemin nasıl çalıştığını göstermek için küçük bir listeyi iter() fonksiyonuyla yineleyiciye
(iterator) çevir ve elemanlarını next() fonksiyonuyla manuel olarak tek tek çek.
4. Kurduğun boru hattını kullanarak loglardan sadece ilk 2 hatayı çekip break ile işlemi tamamen durdur.

In [ ]:
# Test Verisi 
dev_log_verisi = [
        "INFO: Sistem başlatıldı.",
        "WARNING: Bellek kullanımı %80.",
        "ERROR: Veritabanı bağlantısı koptu.",
        "INFO: Yeniden bağlanılıyor...",
        "ERROR: Kimlik doğrulama başarısız (Timeout).",        
        "DEBUG: Kullanıcı oturumu kapatıldı.",
        "ERROR: Disk alanı yetersiz."
    ]

# Generator 1 - Parametre olarak liste alır. Generator döndürür.
def log_okuyucu(log_satirlari):
    index = 0
    while index < len(log_satirlari):
        yield log_satirlari[index]
        index += 1

# Generator 2 - Parametre olarak bir iterator/generator nesnesi alır. Generator döndürür.
def hata_filtresi(okuyucu):
    for log_satiri in okuyucu:
        if "ERROR" in log_satiri:
            yield log_satiri.strip()


# --- Yazdığımız generatorlerin sınanması / Pipeline hata taraması ---
log_okuyucumuz = log_okuyucu(dev_log_verisi)
filtrelenmis_okuyucu = hata_filtresi(log_okuyucumuz)

bulunan_hata = 0
for hata_satiri in filtrelenmis_okuyucu:
    print("Tespit:", hata_satiri)
    bulunan_hata += 1
    if bulunan_hata == 2:
        print("İlk 2 hata bulundu, durduruluyor!")
        break

Tespit: ERROR: Veritabanı bağlantısı koptu.
Tespit: ERROR: Kimlik doğrulama başarısız (Timeout).
İlk 2 hata bulundu, durduruluyor!


### 4 - Iterators ve Generators (3/5)

Bağlam: Bir bankacılık sisteminde, gerçek zamanlı işlem (transaction) akışını analiz ediyorsun. Günlük milyonlarca işlem olduğu için tüm veriyi belleğe yüklemek yerine, işlemleri tek tek, ihtiyaç anında üreten (lazy) bir yapı kurman gerekiyor. Ayrıca belirli bir tutarın üzerindeki işlemleri "şüpheli" olarak işaretleyip ilk birkaçını hızlıca yakalayan bir uyarı sistemi yazacaksın.

Gereksinimler:

1. İşlem tutarlarını içeren bir listeyi yield kullanarak tek tek üreten bir üreteç (generator) fonksiyonu yaz.
2. İlk üreteçten gelen akışı (iterator) girdi olarak alan, yalnızca 10.000 TL'nin üzerindeki tutarları yield eden ikinci bir üreteç yazarak bir "boru hattı" (pipeline) kur.. Bu filtre içinde None veya negatif değerleri de güvenle atlamalısın.
3. Kurduğun boru hattını kullanarak işlem akışından yalnızca ilk 2 şüpheli işlemi yakala, ikinciyi bulduğunda break ile taramayı tamamen durdur (böylece geri kalan veriler hiç işlenmemiş/belleğe alınmamış olsun).
4. Ana programda, küçük bir örnek liste üzerinde iter() ile bir yineleyici oluştur ve next() ile elemanlarını manuel olarak tek tek çek. Yineleyici tükendiğinde fırlatılacak StopIteration hatasını bir try/except bloğuyla yakala.

In [36]:
# Test Verisi
gunluk_islemler = [1500.0, None, -200.0, 12500.0, 8000.0, None, 25000.0, 3000.0, 45000.0] 


def islem_uretici(islemler):
    for islem in islemler:
        yield islem

def supheli_filtre(ham_islemler):
    for islem in ham_islemler:
        if islem is None or islem < 0:
            continue
        if islem > 10_000: 
            yield islem

# --- Tanımladığımız generatorlerin test edilmesi / Pipeline ---
ham_islem_okuyucu = islem_uretici(gunluk_islemler)
supheli_islem_okuyucu = supheli_filtre(ham_islem_okuyucu)

okunan_islem = 0
for supheli_islem in supheli_islem_okuyucu:
    print("Şüpheli islem:", supheli_islem)
    okunan_islem += 1
    if okunan_islem == 2:
        print("Yeterli sayıda şüpheli işlem yakalandı")
        break


# --- Gereksinim 3: Stop Iteration Hatası --- 
gunluk_islemler_iter = iter(gunluk_islemler)
print("\n --- Tüm İşlemler ---")
while True:
    try:
        print(next(gunluk_islemler_iter), end=", ")
    except StopIteration as hata_mesaji:
        print("\nHata mesajı: Yineleyici tükendi, elde edilecek başka eleman kalmadı.")
        break

Şüpheli islem: 12500.0
Şüpheli islem: 25000.0
Yeterli sayıda şüpheli işlem yakalandı

 --- Tüm İşlemler ---
1500.0, None, -200.0, 12500.0, 8000.0, None, 25000.0, 3000.0, 45000.0, 
Hata mesajı: Yineleyici tükendi, elde edilecek başka eleman kalmadı.


### 5 - match-case Yapısı 

Bağlam: Bir mikroservis mimarisinde, dış sistemlerden gelen farklı yapıdaki JSON veri paketlerini (payload) içerdikleri alanlara (keys) ve değerlere göre yönlendiren bir API yönlendiricisi (router) yazıyoruz.
Gereksinimler:
1. Gelen bir sözlüğü (dict) parametre olarak alan ve match ifadesi kullanarak analiz eden bir fonksiyon yaz.
2. İlk case durumunda, gelen veri {"aksiyon": "giris", "kullanici": k} yapısındaysa, kullanıcı adı "admin" olduğunda devreye girecek bir eşleştirme (literal matching) yapıp özel bir yönetici mesajı döndür.
3. İkinci durumda, aynı yapıyı kullanıp k değişkenini yakalayarak (capture) normal kullanıcılar için dinamik bir karşılama mesajı döndür.
4. Üçüncü durumda, {"aksiyon": "veri_cek", "endpoint": e, "limit": l} desenini eşleştirip bir veri çekme onayı döndür.
5. Hiçbir desene uymayan veya eksik alan içeren veriler (edge-case) için case _: (wildcard) kullanarak geçersiz paket uyarısı ver.

In [ ]:
#Test Verisi
istekler = [
    {"aksiyon": "giris", "kullanici": "admin"},                   
    {"aksiyon": "giris", "kullanici": "veri_uzmani_99"},               
    {"aksiyon": "veri_cek", "endpoint": "/sensorler", "limit": 100},     
    {"aksiyon": "sil", "hedef": "kullanici_12"},                  
    {"aksiyon": "giris"}                                         
]

def veri_yonlendirici(veri: dict) -> None:
    match veri:
        case {"aksiyon":"giris", "kullanici": "admin"}:     # Sabit Değer Eşleme
            print("'admin' kullanıcısı 'giris' eylemini gerçekleştirdi.")
        case {"aksiyon":"giris", "kullanici": k}:           # Değişken Eşleme
            print(f"'{k}' kullanıcısı 'giris' eylemini gerçekleştirdi.")
        case {"aksiyon":"veri_cek", "endpoint": e, "limit": l}:        
            print(f"Veri tabebi alındı. '{e}' konumundan '{l}' adet kayıt çekilecek.")
        case _:
            print("Geçersiz paket!")


for veri in istekler:
    veri_yonlendirici(veri)    

'admin' kullanıcısı 'giris' eylemini gerçekleştirdi.
'veri_uzmani_99' kullanıcısı 'giris' eylemini gerçekleştirdi.
Veri tabebi alındı. '/sensorler' konumundan '100' adet kayıt çekilecek.
Geçersiz paket!
Geçersiz paket!
